# 🎯 BNNR — Object Detection Demo (YOLOv8 + COCO128)

**Bulletproof Neural Network Recipe** — Automated Augmentation Search for Detection.

This notebook demonstrates the **complete BNNR detection pipeline** on **Ultralytics COCO128** (128 MS COCO images, ~7 MB auto-download) with **YOLOv8n** weights pretrained on COCO.

- **YOLOv8n** via `UltralyticsDetectionAdapter` (Ultralytics)
- **All 9 BNNR detection augmentation types** — geometric, XAI-driven, composite
- **Bbox-aware** augmentations that automatically update bounding box coordinates
- **DetectionICD / DetectionAICD** — XAI-driven augmentation for detection (unique to BNNR)
- mAP@0.5 and mAP@[.5:.95] metrics with per-class AP
- Detection XAI: attention maps showing where the model focuses per object

### What Makes BNNR Detection Special?

1. **Bbox-aware**: All augmentations that transform geometry (flip, rotate, scale) also update bounding box coordinates, clip them to image bounds, and remove degenerate boxes
2. **DetectionICD / AICD**: Uses bounding box regions as importance priors to create saliency-driven augmentations — unique to BNNR
3. **Mosaic & MixUp**: YOLOv4-style mosaic and alpha-blended mixup with proper box merging
4. **Automatic selection**: BNNR tests each augmentation candidate and keeps only those that improve mAP

### COCO128 dataset

80 COCO classes; images and YOLO labels are fetched with `bnnr.example_data.ensure_coco128_yolo` (same mirror as Ultralytics). **Do not** use `transforms.Normalize()` — BNNR augmentations expect tensors in **[0, 1]**. If you see a warning about values slightly above 1.0, it is usually **colour / MixUp augmentations**, not ImageNet normalization.

⏱ **Runtime:** roughly tens of minutes on a Colab GPU (T4), longer on CPU.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bnnr-team/bnnr/blob/main/examples/detection/bnnr_detection_demo.ipynb)

## 1. Installation

In [ ]:
# Colab: PyPI may lag behind fixes. For current YOLO + scale fixes, prefer:
# %pip install -q "git+https://github.com/bnnr-team/bnnr.git"
%pip install -q bnnr matplotlib ultralytics pyyaml

# If you still see Ultralytics warnings (max 255) or non-finite YOLO loss, run:
# %pip install -q --force-reinstall "git+https://github.com/bnnr-team/bnnr.git"

### Check your `bnnr` install (diagnostic)

Colab often caches Python wheels. If you hit Ultralytics warnings about values up to `255` or **non-finite YOLO loss**, confirm you are on a build that includes the detection fixes:

- `pip install -U bnnr`
- or: `pip install git+https://github.com/bnnr-team/bnnr.git`

## 2. Imports

In [ ]:
import bnnr

print("bnnr version:", getattr(bnnr, "__version__", "?"))
print("bnnr file:", bnnr.__file__)

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torchvision.transforms.functional as TF
import yaml
from PIL import Image
from torch.utils.data import DataLoader, Dataset

from bnnr import BNNRConfig, BNNRTrainer
from bnnr.detection_adapter import UltralyticsDetectionAdapter
from bnnr.detection_augmentations import (
    DetectionHorizontalFlip,
    DetectionVerticalFlip,
    DetectionRandomRotate90,
    DetectionRandomScale,
    MosaicAugmentation,
    DetectionMixUp,
)
from bnnr.detection_collate import detection_collate_fn_with_index
from bnnr.detection_icd import DetectionICD, DetectionAICD
from bnnr.example_data import ensure_coco128_yolo

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
print(f"Using device: {DEVICE}")

## 3. Dataset — COCO128 (YOLO layout)

**COCO128** is a standard 128-image MS COCO subset in YOLO label format. It is downloaded automatically (~7 MB) via `ensure_coco128_yolo` into `./datasets/coco128` (idempotent).

Our `Coco128BnnrDataset` wrapper:
- Resizes images to **640×640** (typical YOLOv8 input)
- Converts YOLO `cls cx cy w h` (normalized) to **xyxy** in pixel coordinates on the resized image
- Uses **class ids 0–79** to match `yolov8n.pt` and `YOLO.predict`
- Returns BNNR's detection format: `(image_tensor, target_dict, index)`

### Data contract

```python
(image_tensor,  # [C, H, W] float32 ∈ [0, 1]
 target_dict,   # {"boxes": [N, 4] xyxy float32, "labels": [N] int64}  labels 0..79
 index)          # int (sample index for caching)
```

**For your own data**, use the same tuple contract with your YOLO or COCO labels — see Section 11 below.

In [ ]:
TARGET_SIZE = 640


def _yolo_label_has_any_line(label_path: Path) -> bool:
    """True if the YOLO label file has at least one ``cls cx cy w h`` row."""
    if not label_path.is_file():
        return False
    for line in label_path.read_text(encoding="utf-8").splitlines():
        parts = line.strip().split()
        if len(parts) >= 5:
            return True
    return False


def _coco80_names() -> list[str]:
    """MS COCO class names in YOLO/COCO id order 0..79 (from Ultralytics cfg)."""
    import ultralytics

    p = Path(ultralytics.__file__).parent / "cfg" / "datasets" / "coco.yaml"
    spec = yaml.safe_load(p.read_text(encoding="utf-8"))
    raw = spec["names"]
    if isinstance(raw, dict):
        return [raw[i] for i in range(80)]
    return [str(x) for x in raw]


class Coco128BnnrDataset(Dataset):
    """COCO128 in YOLO txt format → BNNR (image, target, index). Labels are 0..79."""

    def __init__(self, image_paths: list[Path], coco128_root: Path, target_size: int = 640):
        self.image_paths = image_paths
        self.root = coco128_root
        self.labels_dir = self.root / "labels" / "train2017"
        self.target_size = target_size

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, index: int):
        path = self.image_paths[index]
        img = Image.open(path).convert("RGB")
        w0, h0 = img.size
        img = TF.resize(img, [self.target_size, self.target_size])
        img_tensor = TF.to_tensor(img)

        label_path = self.labels_dir / (path.stem + ".txt")
        boxes: list[list[float]] = []
        labels: list[int] = []
        if label_path.is_file():
            for line in label_path.read_text(encoding="utf-8").splitlines():
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                c = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:5])
                x1o = (cx - bw / 2.0) * w0
                y1o = (cy - bh / 2.0) * h0
                x2o = (cx + bw / 2.0) * w0
                y2o = (cy + bh / 2.0) * h0
                sx = self.target_size / w0
                sy = self.target_size / h0
                x1, y1, x2, y2 = x1o * sx, y1o * sy, x2o * sx, y2o * sy
                x1 = max(0.0, min(float(self.target_size - 1), x1))
                y1 = max(0.0, min(float(self.target_size - 1), y1))
                x2 = max(0.0, min(float(self.target_size), x2))
                y2 = max(0.0, min(float(self.target_size), y2))
                if x2 > x1 and y2 > y1:
                    boxes.append([x1, y1, x2, y2])
                    labels.append(c)

        if not boxes:
            raise RuntimeError(
                f"Coco128BnnrDataset: no valid xyxy boxes after resize/clip for {path}. "
                f"Remove this image from the split or fix labels in {label_path}."
            )

        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.int64),
        }
        return img_tensor, target, index


# ── Download COCO128 and split train / val ────────────────────────────
print("📦 Ensuring Ultralytics COCO128 under ./datasets/coco128 …")
coco_root = Path("./datasets/coco128").resolve()
ensure_coco128_yolo(coco_root)
class_names = _coco80_names()

img_dir = coco_root / "images" / "train2017"
all_jpg = sorted(img_dir.glob("*.jpg"))
if len(all_jpg) < 8:
    raise RuntimeError(f"Expected COCO128 images in {img_dir}, found {len(all_jpg)}.")

labels_dir = coco_root / "labels" / "train2017"
n_train = max(1, int(len(all_jpg) * 0.8))
train_paths = [p for p in all_jpg[:n_train] if _yolo_label_has_any_line(labels_dir / (p.stem + ".txt"))]
val_paths = [p for p in all_jpg[n_train:] if _yolo_label_has_any_line(labels_dir / (p.stem + ".txt"))]
if not train_paths:
    raise RuntimeError(
        "No COCO128 training images with YOLO label rows under labels/train2017. "
        f"Checked {labels_dir}."
    )
if not val_paths:
    raise RuntimeError(
        "No COCO128 validation images with labels — check the dataset or adjust the split."
    )

train_ds = Coco128BnnrDataset(train_paths, coco_root, target_size=TARGET_SIZE)
val_ds = Coco128BnnrDataset(val_paths, coco_root, target_size=TARGET_SIZE)

train_loader = DataLoader(
    train_ds,
    batch_size=4,
    shuffle=True,
    collate_fn=detection_collate_fn_with_index,
    num_workers=0,
)
val_loader = DataLoader(
    val_ds,
    batch_size=4,
    shuffle=False,
    collate_fn=detection_collate_fn_with_index,
    num_workers=0,
)

print(f"Train: {len(train_ds):,}  Val: {len(val_ds):,}  COCO classes: {len(class_names)}")

### Visualize COCO128 samples

Real COCO images with bounding boxes (class ids 0–79).

In [ ]:
# Colour palette (cycled for 80 COCO classes)
COLORS = [
    "#e74c3c", "#2ecc71", "#3498db", "#f39c12", "#9b59b6",
    "#1abc9c", "#e67e22", "#2980b9", "#c0392b", "#27ae60",
    "#8e44ad", "#d35400", "#16a085", "#f1c40f", "#e91e63",
    "#00bcd4", "#795548", "#607d8b", "#ff5722", "#4caf50",
]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
n = len(train_ds)
for i, ax in enumerate(axes.flat):
    idx = min(i * max(1, n // 8), n - 1)
    img, target, _ = train_ds[idx]
    ax.imshow(img.permute(1, 2, 0).numpy())
    for box, label in zip(target["boxes"], target["labels"]):
        label_idx = int(label.item())
        if not (0 <= label_idx < len(class_names)):
            continue
        x1, y1, x2, y2 = box.tolist()
        color = COLORS[label_idx % len(COLORS)]
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                 linewidth=2, edgecolor=color, facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, y1 - 3, class_names[label_idx], fontsize=8,
                color="white", bbox=dict(facecolor=color, alpha=0.8, pad=1))
    ax.set_title(f"Sample idx {idx}", fontsize=10)
    ax.axis("off")
fig.suptitle("COCO128 — Detection Samples", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Model & Adapter

We use **Ultralytics YOLOv8n** (`yolov8n.pt`) — COCO-pretrained weights — wrapped with `UltralyticsDetectionAdapter`:

- **Train mode**: internal YOLO v8 loss from normalized boxes and class ids **0..79**
- **Eval mode**: `YOLO.predict` → prediction dicts aligned with BNNR mAP
- **Metrics**: mAP@0.5 and mAP@[.5:.95] at epoch end

> **Note:** Ground-truth `labels` must match the checkpoint class ids (80-class COCO: **0–79**).

In [ ]:
adapter = UltralyticsDetectionAdapter(
    model_name="yolov8n.pt",
    device=DEVICE,
    num_classes=80,
    lr=1e-3,
)

model = adapter.get_model()
params = sum(p.numel() for p in model.parameters())
print(f"YOLOv8n (Ultralytics): {params:,} parameters")
print("Weights: COCO-pretrained yolov8n.pt")

## 5. Detection Augmentations — All 9 BNNR Types

Detection augmentations are **bbox-aware** — they update bounding box coordinates when transforming images.

| Category | Augmentation | Effect | Bbox Update |
|:---|:---|:---|:---|
| **Geometric** | `DetectionHorizontalFlip` | Flip left-right | Mirrors box x-coordinates |
| **Geometric** | `DetectionVerticalFlip` | Flip top-bottom | Mirrors box y-coordinates |
| **Geometric** | `DetectionRandomRotate90` | Rotate by 90/180/270 | Rotates boxes accordingly |
| **Geometric** | `DetectionRandomScale` | Scale 0.85-1.15x | Scales box coordinates |
| **XAI-driven** | `DetectionICD` | Masks object tiles | Forces context learning |
| **XAI-driven** | `DetectionAICD` | Masks background tiles | Focuses on objects |
| **Composite** | `MosaicAugmentation` | 4-image mosaic | Merges boxes from all 4 |
| **Composite** | `DetectionMixUp` | Alpha-blend 2 images | Combines box targets |
| **Colour** | `AlbumentationsBboxAugmentation` | Brightness/contrast/hue | No box change (colour only) |

BNNR will test each candidate and **automatically keep only those that improve mAP**.

In [ ]:
augmentations = [
    # ---- Geometric (bbox-aware) ----
    DetectionHorizontalFlip(
        probability=0.5, name_override="det_hflip", random_state=SEED,
    ),
    DetectionVerticalFlip(
        probability=0.5, name_override="det_vflip", random_state=SEED + 1,
    ),
    DetectionRandomRotate90(
        probability=0.5, name_override="det_rotate90", random_state=SEED + 2,
    ),
    DetectionRandomScale(
        scale_range=(0.85, 1.15), probability=0.5,
        name_override="det_scale", random_state=SEED + 3,
    ),

    # ---- XAI-driven (unique to BNNR) ----
    DetectionICD(
        probability=0.5, threshold_percentile=70, tile_size=8,
        fill_strategy="gaussian_blur",
        name_override="det_icd", random_state=SEED + 10,
    ),
    DetectionAICD(
        probability=0.5, threshold_percentile=70, tile_size=8,
        fill_strategy="gaussian_blur",
        name_override="det_aicd", random_state=SEED + 11,
    ),

    # ---- Composite ----
    MosaicAugmentation(
        output_size=(TARGET_SIZE, TARGET_SIZE), probability=0.5,
        name_override="det_mosaic", random_state=SEED + 20,
    ),
    DetectionMixUp(
        alpha_range=(0.3, 0.7), probability=0.5,
        name_override="det_mixup", random_state=SEED + 21,
    ),
]

# ---- Albumentations colour jitter (if installed) ----
try:
    import albumentations as alb
    from bnnr.detection_augmentations import AlbumentationsBboxAugmentation

    augmentations.append(
        AlbumentationsBboxAugmentation(
            alb.Compose(
                [
                    alb.RandomBrightnessContrast(p=0.4),
                    alb.HueSaturationValue(p=0.3),
                ],
                bbox_params=alb.BboxParams(
                    format="pascal_voc",
                    label_fields=["labels"],
                    min_visibility=0.2,
                ),
            ),
            name_override="det_color_jitter",
            probability=0.5,
            random_state=SEED + 30,
        )
    )
except ImportError:
    print("(Albumentations not installed — skipping colour jitter candidate)")

print(f"{len(augmentations)} detection augmentation candidates:")
for i, aug in enumerate(augmentations, 1):
    tag = ""
    if "icd" in aug.name or "aicd" in aug.name:
        tag = " [XAI-driven]"
    elif "mosaic" in aug.name or "mixup" in aug.name:
        tag = " [composite]"
    elif "color" in aug.name or "jitter" in aug.name:
        tag = " [colour]"
    else:
        tag = " [geometric]"
    print(f"  {i:2d}. {aug.name:<24s}  p={aug.probability:.2f}{tag}")

### Visualize bbox-aware augmentations on COCO128

See how geometric and XAI-driven augmentations transform images and bounding boxes together. Each demo uses a **fresh NumPy copy** of the boxes so panels do not share mutable state; `DetectionRandomRotate90` always applies a 90°/180°/270° step when it runs (no silent identity).

In [ ]:
# Pick a training sample (CPU numpy copies — avoids shared storage / device issues)
img_t, target, _ = train_ds[0]
img_np = (img_t.permute(1, 2, 0).detach().cpu().numpy() * 255.0).clip(0, 255).astype(np.uint8)
base_tgt = {
    "boxes": target["boxes"].detach().cpu().numpy().astype(np.float32).copy(),
    "labels": target["labels"].detach().cpu().numpy().astype(np.int64).copy(),
}

# Apply 4 different augmentations for comparison (distinct RNG seeds per aug)
demo_augs = [
    ("Original", None),
    ("HFlip", DetectionHorizontalFlip(probability=1.0, random_state=101)),
    ("Rotate90", DetectionRandomRotate90(probability=1.0, random_state=202)),
    ("DetectionICD", DetectionICD(probability=1.0, threshold_percentile=70,
                                  tile_size=8, fill_strategy="gaussian_blur",
                                  random_state=303)),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, (title, aug) in zip(axes, demo_augs):
    if aug is None:
        im, tgt = img_np, {"boxes": base_tgt["boxes"].copy(), "labels": base_tgt["labels"].copy()}
    else:
        im, tgt = aug.apply_with_targets(
            img_np.copy(),
            {"boxes": base_tgt["boxes"].copy(), "labels": base_tgt["labels"].copy()},
        )

    ax.imshow(im)
    boxes = tgt["boxes"] if isinstance(tgt["boxes"], np.ndarray) else tgt["boxes"].numpy()
    labels = tgt["labels"] if isinstance(tgt["labels"], np.ndarray) else tgt["labels"].numpy()
    for box, label in zip(boxes, labels):
        label_idx = int(label)
        if not (0 <= label_idx < len(class_names)):
            continue
        x1, y1, x2, y2 = box
        color = COLORS[label_idx % len(COLORS)]
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                 linewidth=2, edgecolor=color, facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, y1 - 3, class_names[label_idx], fontsize=8,
                color="white", bbox=dict(facecolor=color, alpha=0.8, pad=1))
    ax.set_title(title, fontsize=12)
    ax.axis("off")

fig.suptitle("Bbox-Aware Augmentations — COCO128", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Config

Setting `task="detection"` automatically configures:
- `selection_metric="map_50"` (optimize for mAP@0.5)
- `metrics=["map_50", "map_50_95", "loss"]`
- Detection-specific XAI (activation-based, fast)
- 512x512 preview and XAI images for clear visualization

In [ ]:
config = BNNRConfig(
    task="detection",
    m_epochs=3,               # Epochs per decision branch
    max_iterations=3,         # 3 rounds -> ~12 main-path epochs
    device=DEVICE,
    xai_enabled=True,
    detection_xai_method="activation",
    detection_xai_max_gt_boxes=3,
    detection_xai_max_pred_boxes=3,
    report_dir="reports/detection_coco128_yolov8_demo",
    report_preview_size=512,
    report_xai_size=512,
    save_checkpoints=False,     # Set True for production runs
    early_stopping_patience=2,
    detection_class_names=class_names,
    event_log_enabled=True,
    candidate_pruning_enabled=True,
    candidate_pruning_warmup_epochs=2,
    seed=SEED,
)

print(f"Task:             {config.task}")
print(f"Selection metric: {config.selection_metric}")
print(f"Metrics:          {config.metrics}")
print(f"XAI method:       {config.detection_xai_method}")
print(f"Preview size:     {config.report_preview_size}x{config.report_xai_size}")

## 7. Train

This runs the full BNNR detection pipeline:
1. **Baseline** — fine-tune YOLOv8n without extra augmentations to establish reference mAP
2. **Iteration 1–3** — test each augmentation candidate; keep those that improve mAP@0.5

At each step, BNNR generates:
- mAP@0.5, mAP@[.5:.95], and per-class AP metrics
- Detection XAI attention maps
- Augmentation preview images at 512×512

**Candidate pruning** is enabled: candidates that fall behind early are stopped, saving time.

In [ ]:
trainer = BNNRTrainer(adapter, train_loader, val_loader, augmentations, config)
result = trainer.run()

## 8. Results

In [ ]:
print("=" * 60)
print("  BNNR Detection Results — COCO128 + YOLOv8n")
print("=" * 60)
print(f"  Best path  : {result.best_path}")
print(f"  Metrics    : {result.best_metrics}")
print(f"  Selected   : {result.selected_augmentations}")
print(f"  Report     : {result.report_json_path}")
print("=" * 60)

if result.selected_augmentations:
    print("\nBNNR selected these augmentations (in order of stacking):")
    for i, name in enumerate(result.selected_augmentations, 1):
        tag = ""
        if "icd" in name or "aicd" in name:
            tag = " <- XAI-driven!"
        elif "mosaic" in name or "mixup" in name:
            tag = " <- composite"
        print(f"  {i}. {name}{tag}")
    print("\nAll other candidates were tested but did not improve mAP.")

## 9. Inspect Detection XAI

Detection XAI images show where the model focuses attention for each detected object. Three panels: **Original with boxes | Attention heatmap | Overlay**.

In [ ]:
from IPython.display import display, Image as IPImage
import glob

run_dir = result.report_json_path.parent
xai_files = sorted(glob.glob(str(run_dir / "artifacts" / "xai" / "**" / "*.png"), recursive=True))

if xai_files:
    print(f"Found {len(xai_files)} detection XAI images. Showing first 6:")
    for f in xai_files[:6]:
        print(f"\n{Path(f).name}:")
        display(IPImage(filename=f, width=600))
else:
    print("No XAI images generated.")

# Also show augmentation previews
preview_files = sorted(glob.glob(str(run_dir / "artifacts" / "candidate_previews" / "*.png")))
if not preview_files:
    preview_files = sorted(glob.glob(str(run_dir / "artifacts" / "augmentation_previews" / "*.png")))
if preview_files:
    print(f"\nFound {len(preview_files)} augmentation preview images. Showing first 4:")
    for f in preview_files[:4]:
        print(f"\n{Path(f).name}:")
        display(IPImage(filename=f, width=600))

## 10. Explore the Events Log

Every training event is logged to `events.jsonl`. You can replay them to reconstruct the full training history.

In [ ]:
from bnnr.events import load_events

events_path = run_dir / "events.jsonl"
if events_path.exists():
    events = load_events(events_path)
    event_types = {}
    for e in events:
        t = e.get("type", "unknown")
        event_types[t] = event_types.get(t, 0) + 1

    print(f"Total events: {len(events)}")
    print("\nEvent types:")
    for t, count in sorted(event_types.items()):
        print(f"  {t:<35s} {count}")
else:
    print("No events.jsonl found.")

---

## 11. Using Your Own Data

### Option A: YOLO format (CLI)

```bash
bnnr train -c examples/configs/detection/detection_underwater.yaml \
    --dataset yolo \
    --data-path /path/to/your/data.yaml \
    --with-dashboard
```

### Option B: COCO format (CLI)

```bash
bnnr train -c examples/configs/detection/detection_underwater.yaml \
    --dataset coco_mini \
    --data-path /path/to/coco/dir \
    --with-dashboard
```

### Option C: Python API (full control)

Replace `Coco128BnnrDataset` above with your own dataset class. The only requirement:

```python
def __getitem__(self, idx):
    return (
        image_tensor,  # [C, H, W] float32, values in [0, 1]
        {"boxes": boxes,  # [N, 4] float32, xyxy format
         "labels": labels},  # [N] int64 — must match your detector's class ids
        idx,  # int
    )
```

See the [Custom Data Notebook](../bnnr_custom_data.ipynb) for detailed instructions.

---

## Next Steps

- **Larger YOLO run**: Use `examples/detection/showcase_yolo_coco128.py` with the repo config `examples/configs/detection/yolo_coco128_example.yaml` and `data/coco128/data.yaml` (same auto-download via `ensure_coco128_yolo`).
- **More epochs**: Increase `m_epochs` and `max_iterations` in `BNNRConfig` above for stronger convergence on COCO128.
- **Custom YOLO data**: CLI with `--dataset yolo --data-path your_data.yaml`
- **Classification**: See the [Classification Demo](../classification/bnnr_classification_demo.ipynb)
- **Augmentations**: See the [Augmentations Guide](../bnnr_augmentations_guide.ipynb)
- **Dashboard (local)**: `bnnr dashboard serve --run-dir reports/detection_coco128_yolov8_demo`